<a href="https://colab.research.google.com/github/zulaikha210072/Practical-DataScience-Course_Midterm-Dataviz/blob/main/Midterm_Dataviz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.1 Mount Google Drive (to save your work)
from google.colab import drive
drive.mount('/content/drive')

# 1.2 Install/upgrade required packages
!pip install plotly pandas numpy -q

# 1.3 Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# 1.4 Check versions
print(f"Pandas version: {pd.__version__}")
import plotly
print(f"Plotly version: {plotly.__version__}")
print(f"Numpy version: {np.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pandas version: 2.2.2
Plotly version: 5.24.1
Numpy version: 2.0.2


In [ ]:
# 1. SETUP ENVIRONMENT

# 1.1 Mount Google Drive (optional, to save your work)
from google.colab import drive
drive.mount('/content/drive')

# 1.2 Install/upgrade required packages
!pip install plotly pandas numpy -q

# 1.3 Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# 1.4 Check versions
print(f"Pandas version: {pd.__version__}")
import plotly
print(f"Plotly version: {plotly.__version__}")
print(f"Numpy version: {np.__version__}")

# 2. LOAD DATA

# 2.1 Upload files from your computer
from google.colab import files
print("Please upload your CSV files:")
uploaded = files.upload()  # allows multiple files upload

# 2.2 Load World Bank CSVs using skiprows to avoid extra headers
life_exp = pd.read_csv("API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv", skiprows=4)
gdp = pd.read_csv("API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv", skiprows=4)

# 2.3 Load metadata files (clean CSVs, no skiprows needed)
metadata_country = pd.read_csv("Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv")
metadata_indicator = pd.read_csv("Metadata_Indicator_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv")

# 2.4 Quick look at your data
print("\n📊 Life Expectancy shape:", life_exp.shape)
print("📊 GDP shape:", gdp.shape)
print("📊 Country Metadata shape:", metadata_country.shape)
print("📊 Indicator Metadata shape:", metadata_indicator.shape)

print("\n🔍 First few rows of Life Expectancy:")
print(life_exp.head())

print("\n🔍 First few rows of GDP:")
print(gdp.head())

print("\n🔍 First few rows of Country Metadata:")
print(metadata_country.head())

print("\n🔍 First few rows of Indicator Metadata:")
print(metadata_indicator.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pandas version: 2.2.2
Plotly version: 5.24.1
Numpy version: 2.0.2
Please upload your CSV files:


Saving API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv to API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46 (5).csv
Saving API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv to API_SP.DYN.LE00.IN_DS2_en_csv_v2_128 (3).csv
Saving Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv to Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46 (5).csv
Saving Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv to Metadata_Country_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128 (3).csv
Saving Metadata_Indicator_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv to Metadata_Indicator_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46 (5).csv
Saving Metadata_Indicator_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128.csv to Metadata_Indicator_API_SP.DYN.LE00.IN_DS2_en_csv_v2_128 (3).csv

📊 Life Expectancy shape: (266, 71)
📊 GDP shape: (266, 71)
📊 Country Metadata shape: (265, 6)
📊 Indicator Metadata shape: (1, 4)

🔍 First few rows of Life Expectancy:
                  Country Name Country Code  \
0                        Aruba          ABW   
1  Africa Eastern and Sou

In [ ]:
# Drop unnecessary columns
life_exp = life_exp.drop(columns=['Unnamed: 70'])
gdp = gdp.drop(columns=['Unnamed: 70'])

# Melt
life_long = pd.melt(
    life_exp,
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    var_name='Year',
    value_name='Life_Expectancy'
)
gdp_long = pd.melt(
    gdp,
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    var_name='Year',
    value_name='GDP_per_Capita'
)

# Convert Year to numeric
life_long['Year'] = pd.to_numeric(life_long['Year'], errors='coerce')
gdp_long['Year'] = pd.to_numeric(gdp_long['Year'], errors='coerce')

# Remove rows with missing years
life_long = life_long.dropna(subset=['Year'])
gdp_long = gdp_long.dropna(subset=['Year'])

# Merge life expectancy and GDP
df_merged = pd.merge(
    life_long[['Country Code', 'Year', 'Life_Expectancy']],
    gdp_long[['Country Code', 'Year', 'GDP_per_Capita']],
    on=['Country Code', 'Year'],
    how='inner'
)

# Merge with metadata_country (fix)
df_final = pd.merge(
    df_merged,
    metadata_country[['Country Code', 'Region', 'IncomeGroup', 'TableName']],
    on='Country Code',
    how='left'
)

# Remove missing values
df_final = df_final.dropna(subset=['Life_Expectancy', 'GDP_per_Capita', 'Region'])

# Filter reasonable values
df_final = df_final[
    (df_final['Life_Expectancy'] > 20) &
    (df_final['Life_Expectancy'] < 90) &
    (df_final['GDP_per_Capita'] > 0)
]

# Clean region names
df_final['Region_Clean'] = df_final['Region'].fillna('Other').str.split(',').str[0].str.strip()

# Check final dataset
print(f"\n✅ Final dataset shape: {df_final.shape}")
print(f"✅ Years covered: {df_final['Year'].min()} to {df_final['Year'].max()}")
print(f"✅ Countries/regions: {df_final['Country Code'].nunique()}")
print(f"✅ Total observations: {len(df_final):,}")
print("\n📋 Sample of final data:")
print(df_final.head(10))

# Save cleaned data
df_final.to_csv('/content/cleaned_data.csv', index=False)
print("\n💾 Saved cleaned data to /content/cleaned_data.csv")


✅ Final dataset shape: (11366, 8)
✅ Years covered: 1960 to 2023
✅ Countries/regions: 214
✅ Total observations: 11,366

📋 Sample of final data:
   Country Code  Year  Life_Expectancy  GDP_per_Capita  \
9           ARG  1960        64.242000      778.251707   
13          AUS  1960        70.817073     1813.431099   
14          AUT  1960        68.585610      939.914815   
16          BDI  1960        43.262000       70.905100   
17          BEL  1960        69.701951     1290.286072   
18          BEN  1960        38.775000       89.856925   
19          BFA  1960        36.074000       69.150246   
20          BGD  1960        43.980000       82.481277   
23          BHS  1960        62.454000     1459.253825   
26          BLZ  1960        57.389000      307.414508   

                       Region          IncomeGroup     TableName  \
9   Latin America & Caribbean  Upper middle income     Argentina   
13        East Asia & Pacific          High income     Australia   
14      Europ

In [ ]:
# 4.1 Check data distribution by year
yearly_counts = df_final.groupby('Year').size()
print("📈 Observations per year:")
print(yearly_counts)

# 4.2 Check data by region
region_counts = df_final['Region_Clean'].value_counts()
print("\n🌍 Observations by region:")
print(region_counts)

# 4.3 Basic statistics by region (for recent year)
recent_year = df_final['Year'].max()
recent_data = df_final[df_final['Year'] == recent_year]

print(f"\n📊 Summary for {recent_year} by region:")
region_summary = recent_data.groupby('Region_Clean').agg({
    'Life_Expectancy': ['mean', 'min', 'max'],
    'GDP_per_Capita': ['mean', 'min', 'max']
}).round(1)
print(region_summary)

# 4.4 Find top and bottom countries for recent year
print(f"\n🏆 Top 5 countries by life expectancy ({recent_year}):")
top_life = recent_data.nlargest(5, 'Life_Expectancy')[['TableName', 'Life_Expectancy', 'GDP_per_Capita']]
print(top_life)

print(f"\n⚠️ Bottom 5 countries by life expectancy ({recent_year}):")
bottom_life = recent_data.nsmallest(5, 'Life_Expectancy')[['TableName', 'Life_Expectancy', 'GDP_per_Capita']]
print(bottom_life)

# 4.5 Calculate COVID impact (2020-2021 drop)
covid_data = df_final[df_final['Year'].isin([2019, 2020, 2021])]
covid_pivot = covid_data.pivot_table(
    index=['TableName', 'Country Code'],
    columns='Year',
    values='Life_Expectancy'
).reset_index()

if 2019 in covid_pivot.columns and 2020 in covid_pivot.columns:
    covid_pivot['COVID_Drop'] = covid_pivot[2020] - covid_pivot[2019]
    print("\n🦠 Top 5 countries with largest COVID life expectancy drop:")
    print(covid_pivot.nsmallest(5, 'COVID_Drop')[['TableName', 2019, 2020, 'COVID_Drop']])

📈 Observations per year:
Year
1960    110
1961    113
1962    115
1963    114
1964    114
       ... 
2019    211
2020    210
2021    210
2022    208
2023    203
Length: 64, dtype: int64

🌍 Observations by region:
Region_Clean
Sub-Saharan Africa            2790
Europe & Central Asia         2633
Latin America & Caribbean     2230
East Asia & Pacific           1901
Middle East & North Africa    1256
South Asia                     364
North America                  192
Name: count, dtype: int64

📊 Summary for 2023 by region:
                           Life_Expectancy             GDP_per_Capita  \
                                      mean   min   max           mean   
Region_Clean                                                            
East Asia & Pacific                   73.8  62.1  85.2        18339.3   
Europe & Central Asia                 79.0  70.1  86.4        42351.4   
Latin America & Caribbean             75.0  64.9  81.7        17369.7   
Middle East & North Africa       

In [ ]:
# 5A.1 Filter for meaningful years (every 5 years to keep animation smooth)
years_to_show = list(range(1960, 2024, 5))
df_anim = df_final[df_final['Year'].isin(years_to_show)].copy()

# 5A.2 Create the animated scatter plot
fig = px.scatter(
    df_anim,
    x='GDP_per_Capita',
    y='Life_Expectancy',
    animation_frame='Year',
    animation_group='TableName',
    size='GDP_per_Capita',  # Size by GDP
    color='Region_Clean',
    hover_name='TableName',
    log_x=True,  # CRITICAL: Log scale for GDP
    range_x=[100, 150000],
    range_y=[20, 90],
    labels={
        'GDP_per_Capita': 'GDP per Capita (current US$, log scale)',
        'Life_Expectancy': 'Life Expectancy at Birth (years)',
        'Region_Clean': 'Region'
    },
    title='🌍 Wealth & Health: 60 Years of Global Progress (1960-2023)',
    template='plotly_white'
)

# 5A.3 Customize for better readability
fig.update_traces(
    marker=dict(line=dict(width=1, color='DarkSlateGrey')),
    selector=dict(mode='markers')
)

fig.update_layout(
    title_font_size=20,
    font_family="Arial",
    hovermode='closest',
    legend_title_text='Region',
    xaxis=dict(
        tickvals=[200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000],
        ticktext=['200', '500', '1k', '2k', '5k', '10k', '20k', '50k', '100k']
    ),
    annotations=[
        dict(
            x=0.02,
            y=0.98,
            xref="paper",
            yref="paper",
            text="Source: World Bank",
            showarrow=False,
            font=dict(size=10, color="gray")
        )
    ]
)


fig.show()

# 5A.5 Save as HTML for interactivity
fig.write_html("/content/gapminder_animation.html")
print("✅ Saved interactive chart to /content/gapminder_animation.html")

✅ Saved interactive chart to /content/gapminder_animation.html
